## Check bad annotations

In [10]:
import os, json

import yaml

with open("../configs/maskrcnn_resnet50_fpn_transfer_real.yaml", "r") as f:
    cfg = yaml.safe_load(f)


root = "../" +  cfg["dataset"]["test"]
ann_path = os.path.join(root, "_annotations.coco.json")

with open(ann_path, "r") as f:
    coco = json.load(f)

imgs = {im["id"]: im["file_name"] for im in coco["images"]}

bad = []
for ann in coco["annotations"]:
    seg = ann.get("segmentation", None)
    if seg is None:
        bad.append((ann["id"], ann["image_id"], "missing"))
        continue

    if isinstance(seg, list):
        if len(seg) == 0:
            bad.append((ann["id"], ann["image_id"], "empty list"))
            continue
        # allow [x1,y1,...] or [[...]]
        polys = [seg] if all(isinstance(x, (int, float)) for x in seg) else seg
        polys = [p for p in polys if isinstance(p, list) and len(p) >= 6 and len(p) % 2 == 0]
        if len(polys) == 0:
            bad.append((ann["id"], ann["image_id"], "malformed polygon"))
    elif isinstance(seg, dict):
        if "counts" not in seg:
            bad.append((ann["id"], ann["image_id"], "dict missing counts"))
    else:
        bad.append((ann["id"], ann["image_id"], f"unknown type {type(seg)}"))

print("Bad annotations:", len(bad))
for ann_id, img_id, reason in bad[:20]:
    print(f"- ann_id={ann_id} img={imgs.get(img_id)} reason={reason}")


Bad annotations: 0


## Clean bad Annotations

In [ ]:
# import os, json, shutil

# root = "../" +  cfg["dataset"]["test"]
# ann_path = os.path.join(root, "_annotations.coco.json")

# # backup
# shutil.copy2(ann_path, ann_path.replace(".json", ".bak.json"))

# with open(ann_path, "r") as f:
#     coco = json.load(f)

# before = len(coco["annotations"])
# coco["annotations"] = [a for a in coco["annotations"] if a.get("id") != 100]
# after = len(coco["annotations"])

# with open(ann_path, "w") as f:
#     json.dump(coco, f)

# print("Removed:", before - after, "annotation(s). Now:", after)


Removed: 1 annotation(s). Now: 474
